# Phase 04N: Teacher C → S3 KD Compatibility Entry

Mục tiêu:

1. xác minh Phase 04M đã hoàn thành;
2. khóa Teacher C seed 42 và S3 seed-42 baseline;
3. kiểm tra vocabulary, special IDs và decoder-token indexing;
4. quyết định kênh KD reproduction;
5. khóa logit-KD policy trước GPU preflight;
6. authorize Phase 04O.

Theo Phase 3 contract, Teacher C và S3 cùng dùng shared caption decoder
vocabulary 10.080. Vì vậy direct logit-KD là ứng viên reproduction ưu tiên.
Feature-KD và Sequence-KD vẫn hợp lệ về mặt nghiên cứu nhưng được dành cho
Phase 5 để tránh thay đổi phương pháp reproduction.

Notebook này chỉ đọc metadata và hash file; không huấn luyện, không mở
Held-out Test và không cần GPU.

## Kaggle inputs

- Phase 04M Teacher B → S2 aggregate;
- Phase 03L Teacher C aggregate/teacher pool;
- Phase 03K Teacher C seed-42 full run;
- Phase 02A shared Student bundle;
- Phase 02D S3 seed-42 baseline;
- Phase 01 formal output.


## Cell 1 — Authorization và formal input discovery


In [1]:
import csv
import hashlib
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_c_s3_kd_compatibility_entry"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ID = "P4C_C_S3_LOGIT"
SEED = 42


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


PHASE4M_COMPLETE_PATH = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
    "THREE_SEED_AGGREGATE_COMPLETE.json"
)
ENTRY_AUTHORIZATION_PATH = exactly_one(
    "PHASE4_TEACHER_C_S3_KD_"
    "COMPATIBILITY_ENTRY_AUTHORIZED.json"
)
TEACHER_POOL_COMPLETE_PATH = exactly_one(
    "PHASE3_TEACHER_POOL_COMPLETE.json"
)
PHASE4_REPRODUCTION_AUTHORIZED_PATH = exactly_one(
    "PHASE4_KD_REPRODUCTION_AUTHORIZED.json"
)
TEACHER_C_SEED42_PATH = exactly_one(
    "TEACHER_C_SEED42_FULL_RUN_COMPLETE.json"
)
PHASE2_READY_PATH = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
S3_BASELINE_PATH = exactly_one(
    "S3_SEED42_FULL_RUN_COMPLETE.json"
)
PHASE1_READY_PATH = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)

phase4m = read_json(PHASE4M_COMPLETE_PATH)
entry_authorization = read_json(ENTRY_AUTHORIZATION_PATH)
teacher_pool = read_json(TEACHER_POOL_COMPLETE_PATH)
phase4_reproduction = read_json(
    PHASE4_REPRODUCTION_AUTHORIZED_PATH
)
teacher_seed42 = read_json(TEACHER_C_SEED42_PATH)
phase2_ready = read_json(PHASE2_READY_PATH)
s3_baseline = read_json(S3_BASELINE_PATH)
phase1_ready = read_json(PHASE1_READY_PATH)

assert phase4m["status"] == "complete"
assert phase4m["reproduction_status"] == "passed"
assert phase4m["cider_gate"] == "3 / 3"
assert phase4m["generation_metric_gate"] == "3 / 3"
assert entry_authorization["status"] == "authorized"
assert entry_authorization["source_reproduction_status"] == "passed"
assert entry_authorization["matched_student"] == "S3 — TinyViT-5M"
assert entry_authorization["method_selection_required"] is True
assert teacher_pool["status"] == "complete"
assert teacher_pool["teacher_family_count"] == 3
assert teacher_pool["all_teacher_runs_qualified_for_kd"] is True
assert teacher_pool["phase4_authorized"] is True
assert phase4_reproduction["status"] == "authorized"
assert teacher_seed42["status"] == "complete"
assert teacher_seed42["teacher_id"] == "C"
assert int(teacher_seed42["seed"]) == SEED
assert teacher_seed42["qualified_for_kd"] is True
assert teacher_seed42["quality_gate_passed"] is True
assert phase2_ready["status"] == "ready"
assert "S3" in phase2_ready["student_ids"]
assert s3_baseline["status"] == "complete"
assert s3_baseline["student_id"] == "S3"
assert int(s3_baseline["seed"]) == SEED
assert phase1_ready["status"] == "ready"

for marker in [
    phase4m, entry_authorization, teacher_pool,
    phase4_reproduction, teacher_seed42, s3_baseline,
]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert phase1_ready["test_set_status"] == "sealed"
assert phase1_ready["test_used"] is False

write_json(
    AUDIT_DIR / "phase4n_cell01_inputs.json",
    {
        "audit_version": "phase4n_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "experiment_id": EXPERIMENT_ID,
        "teacher_id": "C",
        "student_id": "S3",
        "development_seed": SEED,
        "phase4m_reproduction": "passed",
        "teacher_pool": "verified",
        "teacher_c_seed42": "qualified_for_kd",
        "s3_seed42_baseline": "verified",
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_payload_read": False,
        "test_parquet_opened": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04N CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher / Student         : C / S3")
print("Development seed          : 42")
print("Phase 04M reproduction    : PASSED")
print("Teacher C pool            : VERIFIED")
print("S3 baseline               : VERIFIED")
print("Dataset/caption read      : False / False")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 5 cells passed")
print("Next                      : verify formal artifacts")


PHASE 04N CELL 1 STATUS
Status                    : PASSED
Teacher / Student         : C / S3
Development seed          : 42
Phase 04M reproduction    : PASSED
Teacher C pool            : VERIFIED
S3 baseline               : VERIFIED
Dataset/caption read      : False / False
Checkpoint payload read   : False
Test Set status           : SEALED
Progress                  : 1 / 5 cells passed
Next                      : verify formal artifacts


## Cell 2 — Manifest, checkpoints và vocabulary artifacts

Hash lại Phase 04M, Teacher C aggregate, Phase 02A và ba checkpoint cần cho
preflight kế tiếp.


In [2]:
CELL_STARTED_AT = time.perf_counter()


def verify_manifest(root, csv_path, sums_path):
    rows = list(csv.DictReader(
        csv_path.open("r", encoding="utf-8", newline="")
    ))
    expected_lines = []
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
        expected_lines.append(
            f'{row["sha256"]}  {row["relative_path"]}'
        )
    actual_lines = [
        line.strip()
        for line in sums_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
    assert actual_lines == expected_lines
    return rows


PHASE4M_ROOT = PHASE4M_COMPLETE_PATH.parent
phase4m_manifest = resolve_inside(
    PHASE4M_ROOT,
    phase4m["manifest"]["csv"]["relative_path"],
)
phase4m_sums = resolve_inside(
    PHASE4M_ROOT,
    phase4m["manifest"]["sha256sums"]["relative_path"],
)
assert sha256_file(phase4m_manifest) == (
    phase4m["manifest"]["csv"]["sha256"]
)
phase4m_rows = verify_manifest(
    PHASE4M_ROOT, phase4m_manifest, phase4m_sums
)
assert len(phase4m_rows) == 13

TEACHER_POOL_ROOT = TEACHER_POOL_COMPLETE_PATH.parent
pool_manifest = resolve_inside(
    TEACHER_POOL_ROOT,
    teacher_pool["manifest"]["csv"]["relative_path"],
)
pool_sums = resolve_inside(
    TEACHER_POOL_ROOT,
    teacher_pool["manifest"]["sha256sums"]["relative_path"],
)
assert sha256_file(pool_manifest) == (
    teacher_pool["manifest"]["csv"]["sha256"]
)
pool_rows = verify_manifest(
    TEACHER_POOL_ROOT, pool_manifest, pool_sums
)
assert len(pool_rows) == 13

PHASE2_ROOT = PHASE2_READY_PATH.parent
phase2_manifest = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["manifests"]["csv"]["relative_path"],
)
phase2_sums = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["manifests"]["sha256sums"]["relative_path"],
)
assert sha256_file(phase2_manifest) == (
    phase2_ready["manifests"]["csv"]["sha256"]
)
phase2_rows = verify_manifest(
    PHASE2_ROOT, phase2_manifest, phase2_sums
)
assert len(phase2_rows) == 38
phase2_by_path = {
    row["relative_path"]: row for row in phase2_rows
}

S3_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s3_encoder_checkpoint"],
)
SHARED_DECODER_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"][
        "shared_decoder_checkpoint"
    ],
)
for checkpoint in [
    S3_INITIAL_CHECKPOINT, SHARED_DECODER_CHECKPOINT
]:
    relative = checkpoint.relative_to(PHASE2_ROOT).as_posix()
    assert relative in phase2_by_path
    assert sha256_file(checkpoint) == (
        phase2_by_path[relative]["sha256"]
    )

registry_path = resolve_inside(
    TEACHER_POOL_ROOT,
    teacher_pool[
        "teacher_c_checkpoint_registry"
    ]["relative_path"],
)
assert sha256_file(registry_path) == (
    teacher_pool["teacher_c_checkpoint_registry"]["sha256"]
)
teacher_registry = read_json(registry_path)
assert teacher_registry["status"] == "locked"
assert teacher_registry["teacher_id"] == "C"
assert teacher_registry["entry_count"] == 3
seed42_registry = [
    item for item in teacher_registry["entries"]
    if int(item["seed"]) == SEED
][0]

TEACHER_C_SEED42_ROOT = TEACHER_C_SEED42_PATH.parent
TEACHER_C_BEST_CHECKPOINT = resolve_inside(
    TEACHER_C_SEED42_ROOT,
    teacher_seed42["best_checkpoint"]["relative_path"],
)
assert sha256_file(TEACHER_C_BEST_CHECKPOINT) == (
    teacher_seed42["best_checkpoint"]["sha256"]
)
assert sha256_file(TEACHER_C_BEST_CHECKPOINT) == (
    seed42_registry["best_checkpoint"]["sha256"]
)

VOCABULARY_PATH = exactly_one(
    "word_vocabulary_minfreq5.json"
)
TOKENIZER_CONTRACT_PATH = exactly_one(
    "tokenizer_contract.json"
)
SEQUENCE_CONTRACT_PATH = exactly_one(
    "sequence_encoding_contract.json"
)
DECODER_CONTRACT_PATH = exactly_one(
    "shared_transformer_decoder_contract.json"
)
vocabulary = read_json(VOCABULARY_PATH)
tokenizer_contract = read_json(TOKENIZER_CONTRACT_PATH)
sequence_contract = read_json(SEQUENCE_CONTRACT_PATH)
decoder_contract = read_json(DECODER_CONTRACT_PATH)
assert vocabulary["vocabulary_size"] == 10080
assert vocabulary["minimum_frequency"] == 5
assert vocabulary["special_token_ids"] == {
    "<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3
}
assert decoder_contract["architecture"]["vocabulary_size"] == 10080

TEACHER_C_COMPATIBILITY_PATH = resolve_inside(
    TEACHER_POOL_ROOT,
    "configs/teacher_c_kd_compatibility_contract.json",
)
teacher_c_compatibility = read_json(
    TEACHER_C_COMPATIBILITY_PATH
)
assert teacher_c_compatibility["status"] == "locked"
assert teacher_c_compatibility["teacher_id"] == "C"
assert teacher_c_compatibility["matched_student_id"] == "S3"

write_json(
    AUDIT_DIR / "phase4n_cell02_artifacts.json",
    {
        "audit_version": "phase4n_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase4m_manifest_rows": len(phase4m_rows),
        "teacher_pool_manifest_rows": len(pool_rows),
        "phase2_manifest_rows": len(phase2_rows),
        "teacher_checkpoint_sha256": sha256_file(
            TEACHER_C_BEST_CHECKPOINT
        ),
        "student_encoder_sha256": sha256_file(
            S3_INITIAL_CHECKPOINT
        ),
        "student_decoder_sha256": sha256_file(
            SHARED_DECODER_CHECKPOINT
        ),
        "shared_vocabulary_size": 10080,
        "checkpoint_payload_read": False,
        "test_parquet_opened": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04N CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 04M manifest        : 13 / 13")
print("Teacher pool manifest     : 13 / 13")
print("Phase 02A manifest        : 38 / 38")
print("Teacher C best checkpoint : VERIFIED")
print("S3 initial checkpoints    : 2 / 2 VERIFIED")
print("Shared vocabulary         : 10080")
print("Special IDs               : 0 / 1 / 2 / 3")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 5 cells passed")
print("Next                      : compatibility decision")


PHASE 04N CELL 2 STATUS
Status                    : PASSED
Phase 04M manifest        : 13 / 13
Teacher pool manifest     : 13 / 13
Phase 02A manifest        : 38 / 38
Teacher C best checkpoint : VERIFIED
S3 initial checkpoints    : 2 / 2 VERIFIED
Shared vocabulary         : 10080
Special IDs               : 0 / 1 / 2 / 3
Checkpoint payload read   : False
Test Set status           : SEALED
Progress                  : 2 / 5 cells passed
Next                      : compatibility decision


## Cell 3 — Output-space compatibility decision

Đối chiếu shared vocabulary, token indexing, chất lượng Teacher/Student và
memory geometry. Chọn direct logit-KD cho reproduction.


In [3]:
CELL_STARTED_AT = time.perf_counter()
entries = teacher_registry["entries"]
assert [int(item["seed"]) for item in entries] == [42, 123, 2026]
for item in entries:
    compatibility = item["kd_compatibility"]
    assert compatibility["direct_logit_kd"]["allowed"] is True
    assert compatibility["sequence_kd"]["allowed"] is True
    assert compatibility["feature_kd"]["allowed"] is True
    assert item["test_set_status"] == "sealed"
    assert item["test_set_used"] is False

locked_compatibility = teacher_c_compatibility["compatibility"]
assert locked_compatibility["direct_logit_kd"]["allowed"] is True
assert locked_compatibility["feature_kd"]["allowed"] is True
assert locked_compatibility["sequence_kd"]["allowed"] is True

TEACHER_VOCABULARY_SIZE = 10080
STUDENT_VOCABULARY_SIZE = int(
    vocabulary["vocabulary_size"]
)
assert TEACHER_VOCABULARY_SIZE == STUDENT_VOCABULARY_SIZE
SPECIAL_IDS = {"pad": 0, "bos": 1, "eos": 2, "unk": 3}

teacher_cider = float(
    teacher_seed42["best_result"]["metrics"]["CIDEr-D"]
)
student_cider = float(
    s3_baseline["best_validation_metrics"]["CIDEr-D"]
)
quality_margin = teacher_cider - student_cider
assert quality_margin > 0.001

COMPATIBILITY_DECISION = {
    "schema_version": "phase4n_kd_compatibility_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "experiment_id": EXPERIMENT_ID,
    "teacher": {
        "id": "C",
        "model": "SigLIP ViT-B/16 + shared caption decoder",
        "seed": 42,
        "vocabulary_size": TEACHER_VOCABULARY_SIZE,
        "special_token_ids": SPECIAL_IDS,
        "expected_logits": "[B,T,10080]",
        "visual_memory": "[B,196,256]",
        "validation_cider_d": teacher_cider,
    },
    "student": {
        "id": "S3",
        "model": "TinyViT-5M + shared Transformer decoder",
        "seed": 42,
        "vocabulary_size": STUDENT_VOCABULARY_SIZE,
        "special_token_ids": SPECIAL_IDS,
        "expected_logits": "[B,T,10080]",
        "visual_memory": "[B,49,256]",
        "validation_cider_d": student_cider,
    },
    "quality_margin_cider_d": quality_margin,
    "methods": {
        "direct_logit_kd": {
            "status": "selected_for_phase4_reproduction",
            "reason": (
                "Teacher and Student share vocabulary size, special "
                "token IDs, decoder-token indexing and output dimension."
            ),
        },
        "feature_kd": {
            "status": "deferred_to_phase5_improvement",
            "reason": (
                "Teacher has 196 visual tokens and S3 has 49; an "
                "explicit spatial alignment/projection is required."
            ),
        },
        "sequence_kd": {
            "status": "deferred_to_phase5_ablation",
            "reason": (
                "Valid but discards the richer aligned probability "
                "distribution available for direct logit KD."
            ),
        },
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
COMPATIBILITY_PATH = CONFIG_DIR / (
    "teacher_c_s3_kd_compatibility_contract.json"
)
write_json(COMPATIBILITY_PATH, COMPATIBILITY_DECISION)

write_json(
    AUDIT_DIR / "phase4n_cell03_compatibility.json",
    {
        "audit_version": "phase4n_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_vocabulary_size": TEACHER_VOCABULARY_SIZE,
        "student_vocabulary_size": STUDENT_VOCABULARY_SIZE,
        "special_token_ids_exact": True,
        "logit_dimension_exact": True,
        "direct_logit_kd": "selected",
        "sequence_kd": "deferred",
        "feature_kd": "deferred",
        "teacher_student_cider_margin": quality_margin,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04N OUTPUT-SPACE COMPATIBILITY")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher/Student vocabulary: 10080 / 10080")
print("Special IDs exact         : True")
print("Logit dimensions          : [B,T,10080] / [B,T,10080]")
print("Direct Logit-KD           : SELECTED")
print("Feature KD                : DEFERRED TO PHASE 5")
print("Sequence KD               : DEFERRED TO PHASE 5")
print("Teacher CIDEr-D           :", teacher_cider)
print("S3 baseline CIDEr-D       :", student_cider)
print("Quality margin            :", quality_margin)
print("Test Set status           : SEALED")
print("Progress                  : 3 / 5 cells passed")
print("Next                      : lock logit-KD policy")


PHASE 04N OUTPUT-SPACE COMPATIBILITY
Status                    : PASSED
Teacher/Student vocabulary: 10080 / 10080
Special IDs exact         : True
Logit dimensions          : [B,T,10080] / [B,T,10080]
Direct Logit-KD           : SELECTED
Feature KD                : DEFERRED TO PHASE 5
Sequence KD               : DEFERRED TO PHASE 5
Teacher CIDEr-D           : 1.2558454959666379
S3 baseline CIDEr-D       : 1.0767986187427856
Quality margin            : 0.17904687722385226
Test Set status           : SEALED
Progress                  : 3 / 5 cells passed
Next                      : lock logit-KD policy


## Cell 4 — Logit-KD reproduction policy

Khóa temperature, alpha, mask, augmentation và các cổng bắt buộc cho GPU
preflight. Teacher luôn frozen/eval/no-grad.


In [4]:
CELL_STARTED_AT = time.perf_counter()
LOGIT_KD_POLICY = {
    "schema_version": "phase4n_logit_kd_policy_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "experiment_id": EXPERIMENT_ID,
    "teacher": {
        "id": "C",
        "model": "google/siglip-base-patch16-224 + shared decoder",
        "revision": seed42_registry["revision"],
        "seed": 42,
        "checkpoint_sha256": sha256_file(
            TEACHER_C_BEST_CHECKPOINT
        ),
        "mode": "eval_frozen_no_grad",
        "visual_memory": "[B,196,256]",
        "logits": "[B,T,10080]",
    },
    "student": {
        "id": "S3",
        "model": "TinyViT-5M + shared Transformer decoder",
        "initial_encoder_sha256": sha256_file(
            S3_INITIAL_CHECKPOINT
        ),
        "initial_decoder_sha256": sha256_file(
            SHARED_DECODER_CHECKPOINT
        ),
        "visual_memory": "[B,49,256]",
        "logits": "[B,T,10080]",
        "vocabulary_size": 10080,
        "maximum_lexical_tokens": 30,
        "maximum_decoder_steps": 31,
    },
    "supervision": {
        "human_caption": (
            "one cyclic Train+Restval caption per image per epoch"
        ),
        "loss": (
            "L = (1-alpha) * CE_human + alpha * T^2 * "
            "KL(teacher_T || student_T)"
        ),
        "temperature": 4.0,
        "alpha": 0.5,
        "mask": "labels != PAD_ID",
        "teacher_logits_detached": True,
        "shared_geometry": True,
        "model_specific_normalization": True,
    },
    "data_boundary": {
        "training_split": "Train+Restval only",
        "validation_used_for_selection_only": True,
        "held_out_test_opened": False,
        "test_set_status": "sealed",
    },
    "phase04o_preflight": {
        "fixed_sample_images": 64,
        "required_checks": [
            "strict checkpoint loads",
            "teacher and student logits shape exact",
            "PAD mask exact",
            "CE, KL and combined loss finite",
            "teacher frozen and gradient-free",
            "student gradients finite and nonzero",
            "one-batch overfit",
            "discarded optimizer-step pilot",
            "exact checkpoint/resume",
        ],
        "training_started": False,
    },
    "deferred_methods": {
        "feature_kd": "Phase 5 improved KD",
        "sequence_kd": "Phase 5 ablation",
    },
}
LOGIT_POLICY_PATH = CONFIG_DIR / (
    "teacher_c_s3_logit_kd_reproduction_policy.json"
)
write_json(LOGIT_POLICY_PATH, LOGIT_KD_POLICY)

write_json(
    AUDIT_DIR / "phase4n_cell04_logit_policy.json",
    {
        "audit_version": "phase4n_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "temperature": 4.0,
        "alpha": 0.5,
        "teacher_frozen": True,
        "teacher_logits_detached": True,
        "shared_vocabulary": True,
        "validation_used_for_selection_only": True,
        "feature_kd_deferred": True,
        "sequence_kd_deferred": True,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04N LOGIT-KD POLICY")
print("=" * 72)
print("Status                    : LOCKED")
print("Temperature / alpha       : 4.0 / 0.5")
print("Teacher mode              : EVAL / FROZEN / NO-GRAD")
print("Teacher logits detached   : True")
print("Mask                      : labels != PAD_ID")
print("Train split               : Train+Restval only")
print("Validation                : selection/evaluation only")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 5 cells passed")
print("Next                      : package and authorize Phase 04O")


PHASE 04N LOGIT-KD POLICY
Status                    : LOCKED
Temperature / alpha       : 4.0 / 0.5
Teacher mode              : EVAL / FROZEN / NO-GRAD
Teacher logits detached   : True
Mask                      : labels != PAD_ID
Train split               : Train+Restval only
Validation                : selection/evaluation only
Training started          : False
Test Set status           : SEALED
Progress                  : 4 / 5 cells passed
Next                      : package and authorize Phase 04O


## Cell 5 — Formal package và Phase 04O authorization


In [5]:
CELL_STARTED_AT = time.perf_counter()
NEXT_AUTHORIZATION = OUTPUT_ROOT / (
    "PHASE4_TEACHER_C_S3_LOGIT_KD_PREFLIGHT_AUTHORIZED.json"
)
write_json(
    NEXT_AUTHORIZATION,
    {
        "marker_version": (
            "phase4_teacher_c_s3_logit_kd_"
            "preflight_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "experiment_id": EXPERIMENT_ID,
        "teacher_id": "C",
        "student_id": "S3",
        "development_seed": 42,
        "selected_method": "direct_logit_kd",
        "temperature": 4.0,
        "alpha": 0.5,
        "policy": {
            "relative_path": LOGIT_POLICY_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(LOGIT_POLICY_PATH),
        },
        "next_phase": (
            "Phase 04O Teacher C -> S3 Logit-KD GPU preflight"
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

MANIFEST_CSV = MANIFEST_DIR / "phase4n_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4n_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4n_SHA256SUMS.txt"
FINAL_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_C_S3_KD_"
    "COMPATIBILITY_ENTRY_COMPLETE.json"
)
artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, FINAL_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4n_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(
        OUTPUT_ROOT, row["relative_path"]
    )
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

write_json(
    FINAL_MARKER,
    {
        "marker_version": (
            "phase4_teacher_c_s3_kd_"
            "compatibility_entry_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "experiment_id": EXPERIMENT_ID,
        "teacher": "Teacher C — SigLIP ViT-B/16",
        "student": "S3 — TinyViT-5M",
        "development_seed": 42,
        "compatibility_decision": {
            "relative_path": COMPATIBILITY_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(COMPATIBILITY_PATH),
        },
        "selected_method": "direct_logit_kd",
        "feature_kd": "deferred_to_phase5",
        "sequence_kd": "deferred_to_phase5",
        "phase04o_authorized": True,
        "next_authorization": {
            "relative_path": NEXT_AUTHORIZATION.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(NEXT_AUTHORIZATION),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04N CELL 5 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher / Student         : C / S3")
print("Selected reproduction KD  : DIRECT LOGIT-KD")
print("Temperature / alpha       : 4.0 / 0.5")
print("Feature/Sequence KD       : DEFERRED TO PHASE 5")
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Phase 04O GPU preflight   : AUTHORIZED")
print("Progress                  : 5 / 5 cells passed")
print("Marker                    :", FINAL_MARKER)


PHASE 04N CELL 5 STATUS
Status                    : COMPLETE
Teacher / Student         : C / S3
Selected reproduction KD  : DIRECT LOGIT-KD
Temperature / alpha       : 4.0 / 0.5
Feature/Sequence KD       : DEFERRED TO PHASE 5
Manifest rows             : 7
Manifest verification     : ALL PASSED
Training started          : False
Test Set status           : SEALED
Phase 04O GPU preflight   : AUTHORIZED
Progress                  : 5 / 5 cells passed
Marker                    : /kaggle/working//phase4_teacher_c_s3_kd_compatibility_entry/PHASE4_TEACHER_C_S3_KD_COMPATIBILITY_ENTRY_COMPLETE.json
